In [ ]:


import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, roc_curve)
import warnings
warnings.filterwarnings('ignore')

# ============================================================

# ============================================================
FILE_NAME      = "FINAL_CLEAN_FULL_with_demo_apoe_age_fixed (1) (1).csv"
HOLDOUT_SITE   = "130"          
ID_COL         = 'PTID'
TIME_COL       = 'EXAMDATE'
TARGET_COL     = 'DIAGNOSIS'
MAX_SEQ_LENGTH = 15
SEED           = 42


FEATURE_COLS = ['MMSCORE', 'TOTAL13', 'AGE', 'EDUCATION', 'SEX_BIN', 'APOE4_carrier']

# ============================================================

# ============================================================
print("=" * 60)
print("STEP 1: Loading data...")
print("=" * 60)

df = pd.read_csv(FILE_NAME)
df['SITE'] = df['PTID'].str[:3]

print(f"Total rows: {len(df)}")
print(f"Total patients: {df['PTID'].nunique()}")
print(f"Holdout site '{HOLDOUT_SITE}' patients: {df[df['SITE']==HOLDOUT_SITE]['PTID'].nunique()}")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Applying Biological Irreversibility Filter (BIF)...")
print("=" * 60)

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
df = df.dropna(subset=[TIME_COL, TARGET_COL, ID_COL])
df = df.sort_values([ID_COL, TIME_COL])


reversals_before = 0
for ptid, group in df.groupby(ID_COL):
    diag = group[TARGET_COL].values
    for i in range(1, len(diag)):
        if diag[i] < diag[i-1]:
            reversals_before += 1


df[TARGET_COL] = df.groupby(ID_COL)[TARGET_COL].transform('cummax')


reversals_after = 0
for ptid, group in df.groupby(ID_COL):
    diag = group[TARGET_COL].values
    for i in range(1, len(diag)):
        if diag[i] < diag[i-1]:
            reversals_after += 1

print(f"Reversals BEFORE BIF: {reversals_before}")
print(f"Reversals AFTER  BIF: {reversals_after}")
print(f"BIF corrected {reversals_before - reversals_after} records")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Splitting training vs holdout site...")
print("=" * 60)

df_holdout  = df[df['SITE'] == HOLDOUT_SITE].copy()
df_train_all = df[df['SITE'] != HOLDOUT_SITE].copy()

print(f"Training sites patients : {df_train_all['PTID'].nunique()}")
print(f"Holdout site patients   : {df_holdout['PTID'].nunique()}")
print(f"Holdout site rows       : {len(df_holdout)}")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Imputing and scaling features...")
print("=" * 60)

for col in FEATURE_COLS:
    if df_train_all[col].isnull().any():
        fill_val = df_train_all[col].mean()
        df_train_all[col] = df_train_all[col].fillna(fill_val)
        df_holdout[col]   = df_holdout[col].fillna(fill_val)  # use training mean


numeric_features = ['MMSCORE', 'TOTAL13', 'AGE', 'EDUCATION']
scaler = MinMaxScaler()
df_train_all[numeric_features] = scaler.fit_transform(df_train_all[numeric_features])
df_holdout[numeric_features]   = scaler.transform(df_holdout[numeric_features])

print("Scaling done. Scaler fitted on training data only.")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Building sequences...")
print("=" * 60)

def build_sequences(data):
    X_data, y_data, ptids = [], [], []
    data = data.sort_values([ID_COL, TIME_COL])
    # Shift labels: 1->0, 2->1, 3->2
    data = data.copy()
    data[TARGET_COL] = data[TARGET_COL] - 1.0
    for ptid, group in data.groupby(ID_COL):
        X_seq  = group[FEATURE_COLS].values
        y_label = int(group[TARGET_COL].iloc[-1])
        X_data.append(X_seq)
        y_data.append(y_label)
        ptids.append(ptid)
    return np.array(X_data, dtype=object), np.array(y_data, dtype=np.int32), ptids

X_train_seq, y_train_seq, train_ptids = build_sequences(df_train_all)
X_hold_seq,  y_hold_seq,  hold_ptids  = build_sequences(df_holdout)

NUM_CLASSES = 3

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LENGTH,
                             dtype='float32', padding='post')
X_hold_pad  = pad_sequences(X_hold_seq,  maxlen=MAX_SEQ_LENGTH,
                             dtype='float32', padding='post')

y_train_oh  = to_categorical(y_train_seq, num_classes=NUM_CLASSES)
y_hold_oh   = to_categorical(y_hold_seq,  num_classes=NUM_CLASSES)

print(f"Training sequences : {X_train_pad.shape}")
print(f"Holdout sequences  : {X_hold_pad.shape}")
print(f"Features per step  : {X_train_pad.shape[2]}")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Building and training Bi-GRU model on training sites...")
print("=" * 60)

NUM_FEATURES = X_train_pad.shape[2]

# Class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train_seq), y=y_train_seq)
class_weight_dict = {i: cw[i] for i in range(len(cw))}
print("Class weights:", class_weight_dict)

def build_bigru_model(num_features, max_seq_len, num_classes):
    model = Sequential([
        Bidirectional(GRU(units=64, activation='tanh',
                          return_sequences=False),
                      input_shape=(max_seq_len, num_features)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

tf.random.set_seed(SEED)
model = build_bigru_model(NUM_FEATURES, MAX_SEQ_LENGTH, NUM_CLASSES)
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train_oh,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Evaluating on holdout site 130 (NEVER seen during training)...")
print("=" * 60)

# Get probabilities
y_hold_prob = model.predict(X_hold_pad)
y_hold_pred = np.argmax(y_hold_prob, axis=1)
y_hold_true = np.argmax(y_hold_oh,   axis=1)

# Training performance for comparison
y_train_prob = model.predict(X_train_pad)
y_train_pred = np.argmax(y_train_prob, axis=1)
y_train_true = np.argmax(y_train_oh,   axis=1)

def compute_metrics(y_true, y_prob, y_pred, dataset_name):
    print(f"\n--- {dataset_name} ---")
    # AUC (macro OvR)
    try:
        auc = roc_auc_score(to_categorical(y_true, NUM_CLASSES),
                             y_prob, multi_class='ovr', average='macro')
        print(f"AUC (macro OvR): {auc:.4f}")
    except Exception as e:
        print(f"AUC error: {e}")
        auc = None

    # Classification report
    print(classification_report(
        y_true, y_pred,
        target_names=['CN (0)', 'MCI (1)', 'AD (2)'],
        digits=4
    ))


    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(cm)
    return auc

train_auc = compute_metrics(y_train_true, y_train_prob, y_train_pred, "TRAINING SITES")
hold_auc  = compute_metrics(y_hold_true,  y_hold_prob,  y_hold_pred,  "HOLDOUT SITE 130")

# ============================================================

# ============================================================
print("\n" + "=" * 60)
print("STEP 8: SUMMARY — Copy this table into your paper")
print("=" * 60)

from sklearn.metrics import recall_score, precision_score, f1_score

def get_summary(y_true, y_pred, y_prob, name, n_patients):
    try:
        auc = roc_auc_score(to_categorical(y_true, NUM_CLASSES),
                             y_prob, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    sens = recall_score(y_true, y_pred, average='macro')
    spec_list = []
    for cls in range(NUM_CLASSES):
        tp = np.sum((y_true == cls) & (y_pred == cls))
        tn = np.sum((y_true != cls) & (y_pred != cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    spec = np.mean(spec_list)
    f1   = f1_score(y_true, y_pred, average='macro')
    print(f"| {name:<30} | {n_patients:>10} | {auc:>6.4f} | {sens*100:>11.2f}% | {spec*100:>11.2f}% | {f1:>6.4f} |")

print(f"\n{'Dataset':<32} {'N Patients':>10} {'AUC':>6} {'Sensitivity':>12} {'Specificity':>12} {'F1':>6}")
print("-" * 85)
get_summary(y_train_true, y_train_pred, y_train_prob,
            "Training sites (74 sites)", df_train_all['PTID'].nunique())
get_summary(y_hold_true,  y_hold_pred,  y_hold_prob,
            "Holdout site 130 (external)", df_holdout['PTID'].nunique())

print("\n" + "=" * 60)
print("DONE. Copy the table above into your Results section.")
print("Save this output — you will need these numbers for your paper.")
print("=" * 60)


STEP 1: Loading data...
Total rows: 15168
Total patients: 3788
Holdout site '130' patients: 98

STEP 2: Applying Biological Irreversibility Filter (BIF)...
Reversals BEFORE BIF: 199
Reversals AFTER  BIF: 0
BIF corrected 199 records

STEP 3: Splitting training vs holdout site...
Training sites patients : 3664
Holdout site patients   : 98
Holdout site rows       : 388

STEP 4: Imputing and scaling features...
Scaling done. Scaler fitted on training data only.

STEP 5: Building sequences...
Training sequences : (3664, 15, 6)
Holdout sequences  : (98, 15, 6)
Features per step  : 6

STEP 6: Building and training Bi-GRU model on training sites...
Class weights: {0: np.float64(0.8927875243664717), 1: np.float64(0.9344554960469268), 2: np.float64(1.234917425008426)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        27,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,875 (124.51 KB)

 Trainable params: 31,875 (124.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.4562 - loss: 1.0235 - val_accuracy: 0.4196 - val_loss: 1.0048
Epoch 2/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5678 - loss: 0.8110 - val_accuracy: 0.4523 - val_loss: 0.9217
Epoch 3/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6187 - loss: 0.7516 - val_accuracy: 0.4605 - val_loss: 0.8877
Epoch 4/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.6439 - loss: 0.7188 - val_accuracy: 0.5123 - val_loss: 0.8527
Epoch 5/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.6554 - loss: 0.6955 - val_accuracy: 0.5204 - val_loss: 0.8573
Epoch 6/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6621 - loss: 0.6880 - val_accuracy: 0.5477 - val_loss: 0.8309
Epoch 7/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6615 - loss: 0.6730 - val_accuracy: 0.5504 - val_loss: 0.8390
Epoch 8/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6630 - loss: 0.6663 - val_accu